In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="../data/chroma_db"
)

In [4]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [5]:
consulta = "Cuanto pagará el empleador al trabajador JULIÁN MIGUEL TORRES CALVO segun se indica en QUINTA. Retribución?"

resultados = retriever.invoke(consulta)

print("Top 1:\n")
for i, doc in enumerate(resultados, start=4):
    print(f"Contenido: {doc.page_content}")
    print(f"Metadatos: {doc.metadata}")

Top 1:

Contenido: DNI: 62491837-T.
Nacionalidad: Española.
Fecha de nacimiento: 11 de junio de 1993.
Estado civil: Soltero.
Domicilio: Calle Jardines del Turia nº 18, Piso 4, Puerta A, 46010, Valencia, España.
Correo electrónico: julian.torres@atlantis-ejemplo.es
Teléfono: +34 655 772 309.
II. DECLARACIONES
A. Declara el Empleador:
Que está legalmente constituido y facultado para contratar personal.
Que necesita reforzar su equipo técnico especializado.
B. Declara el Trabajador:
Que posee la cualificación técnica adecuada para el puesto.
Que acepta voluntariamente la relación laboral en régimen de subordinación.
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
• 
1. 
2. 
1. 
2. 
1
Metadatos: {'source': '..\\data\\contratos_laborales\\Contrato Laboral Individual II.pdf', 'page': 0, 'producer': 'WeasyPrint 65.1', 'title': 'Contrato Laboral Individual Iv', 'total_pages': 3, 'author': 'ChatGPT Canvas', 'creationdate': '', 'creator': 'ChatGPT', 'page_label': '1'}
Contenido: CONTRATO DE TRABAJO PO

In [6]:
from dotenv import load_dotenv
import os

In [7]:
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import json

In [9]:
# Modelo que quieres usar
MODEL = "gemini-2.5-flash-lite"

In [14]:
chat = ChatGoogleGenerativeAI(model=MODEL, temperature=0, google_api_key=API_KEY)

plantilla = PromptTemplate(
    input_variables=["documento", "pregunta"],
    template =  """
        Usa la información del siguiente documento para responder la pregunta de manera clara y concisa.

        Documento:
        {documento}

        Pregunta:
        {pregunta}

        Respuesta: en formato JSON con los siguientes campos: salario_bruto, salario_bruto_mensual, retribucion_variable
        """
)

In [15]:
chain_llm = plantilla | chat

if resultados:
    # Tomar hasta los 4 primeros resultados
    primeros_4 = resultados[:4]
    
    # Concatenar el contenido de los documentos
    contenido_concatenado = "\n\n---\n\n".join(doc.page_content for doc in primeros_4)

In [16]:
respuesta = chain_llm.invoke({
    "documento": contenido_concatenado,
    "pregunta": consulta
})

In [17]:
print("Respuesta del LLM:\n", respuesta)

Respuesta del LLM:
 content='```json\n{\n  "salario_bruto": "42.000 € anuales",\n  "salario_bruto_mensual": "3.500 € (12 pagas)",\n  "retribucion_variable": "Hasta un 8% anual según objetivos"\n}\n```' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c00b5-15ad-7913-a49d-00b97533909a-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1181, 'output_tokens': 72, 'total_tokens': 1253, 'input_token_details': {'cache_read': 0}}
